# 00 · Shared setup: storage, table names and Delta helpers

**Purpose.** One place for everything the other notebooks share: where the data lives, what each Delta table is called, how tables are written and verified, and how models are split and scored. Keeping it here means all team members write to the same tables in the same way.

**How it is used.** This notebook is not run on its own. Every other notebook starts with a cell containing only `%run ./00_setup`, which runs these cells and makes the names and functions below available.

**Inputs.** None.

**Outputs.** No tables. It creates the schema `workspace.car_insurance` and the volumes `raw` (source files) and `models` (saved Spark ML models) if they do not exist yet.

**Assumptions**

- The catalog is named `workspace` (the Databricks Free Edition default). If yours differs, change `CATALOG` below; nothing else needs editing.
- Serverless compute.

**What it provides**

| Name | Use |
|---|---|
| `BRONZE_TABLE`, `SILVER_TABLE`, `GOLD_FEATURES_TABLE`, `GOLD_SUMMARY_TABLE`, `SPLIT_TABLE` | Full table names. Always use these instead of typing a table name |
| `CLAIMS_CSV`, `SPLIT_CSV`, `MODEL_DIR` | Paths inside the volumes |
| `require_source_files(...)` | Stops with a clear message if an uploaded file is missing |
| `write_delta(df, table_name)` | Saves a DataFrame as a Delta table |
| `validate_tables({...})` | Reads tables back and checks format, rows, columns and key |
| `load_model_splits()` | Train, validation and test DataFrames with the Phase 2 split |
| `evaluate_classifier(predictions)` | Accuracy, precision, recall, F1, ROC-AUC and confusion counts |

In [ ]:
# --- Where everything lives ---------------------------------------------------
CATALOG = "workspace"
SCHEMA = "car_insurance"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}"
RAW_DIR = f"{VOLUME_ROOT}/raw"          # uploaded source files
MODEL_DIR = f"{VOLUME_ROOT}/models"     # saved Spark ML models

CLAIMS_CSV = f"{RAW_DIR}/Car_Insurance_Claim.csv"
SPLIT_CSV = f"{RAW_DIR}/split_assignment.csv"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.claims_silver"
GOLD_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_features"
GOLD_SUMMARY_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_summary"
SPLIT_TABLE = f"{CATALOG}.{SCHEMA}.claims_model_split"

ID_COLUMN = "ID"            # one row per driver in every row-level table
LABEL_COLUMN = "OUTCOME"    # 1 = the driver filed a claim

# The 19 columns of the source file, in file order.
SOURCE_COLUMNS = [
    "ID", "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
    "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "MARRIED", "CHILDREN",
    "POSTAL_CODE", "ANNUAL_MILEAGE", "VEHICLE_TYPE", "SPEEDING_VIOLATIONS",
    "DUIS", "PAST_ACCIDENTS", "OUTCOME",
]

In [ ]:
# --- Table contract -----------------------------------------------------------
# What each table is, which notebook writes it, and what the next notebook relies on.
# `required_columns` is the minimum: a notebook may add more columns.
# `key` must be unique: it is how tables are joined to each other and to the split.
TABLE_CONTRACT = {
    BRONZE_TABLE: {
        "layer": "Bronze", "written_by": "01_bronze_layer",
        "description": "Raw source records with original fields and values, plus ingestion metadata",
        "required_columns": SOURCE_COLUMNS, "key": None,
    },
    SILVER_TABLE: {
        "layer": "Silver", "written_by": "02_silver_layer",
        "description": "Cleaned and standardized drivers, one row per ID",
        "required_columns": SOURCE_COLUMNS, "key": ID_COLUMN,
    },
    GOLD_FEATURES_TABLE: {
        "layer": "Gold", "written_by": "03_gold_layer",
        "description": "Model-ready drivers: ID, OUTCOME and the input features, one row per ID",
        "required_columns": [ID_COLUMN, LABEL_COLUMN], "key": ID_COLUMN,
    },
    GOLD_SUMMARY_TABLE: {
        "layer": "Gold", "written_by": "03_gold_layer",
        "description": "Aggregated claim metrics that answer the project question",
        "required_columns": [], "key": None,
    },
    SPLIT_TABLE: {
        "layer": "Reference", "written_by": "03b_delta_tables",
        "description": "Phase 2 train, validation or test assignment for each driver ID",
        "required_columns": [ID_COLUMN, "split"], "key": ID_COLUMN,
    },
}

In [ ]:
# --- Storage ------------------------------------------------------------------
import os

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.models")

# Spark ML on serverless compute needs a volume path for its temporary files.
# Databricks error messages and forum answers use two spellings, so both are set.
ML_TEMP_DIR = f"{MODEL_DIR}/_tmp"
try:
    os.makedirs(ML_TEMP_DIR, exist_ok=True)
except OSError:
    pass
os.environ["SPARKML_TEMP_DFS_PATH"] = ML_TEMP_DIR
os.environ["SPARK_ML_TMP_DFS_PATH"] = ML_TEMP_DIR


def require_source_files(*paths):
    """Stop early, with instructions, if a file has not been uploaded to the volume."""
    def exists(path):
        try:
            dbutils.fs.ls(path)
            return True
        except Exception:
            return False

    missing = [path for path in paths if not exists(path)]
    if missing:
        raise FileNotFoundError(
            "Upload these files (Catalog > workspace > car_insurance > raw > "
            "'Upload to this volume'), then run the notebook again:\n  " + "\n  ".join(missing))
    print("Found:", ", ".join(paths))

In [ ]:
# --- Writing and validating Delta tables ----------------------------------------
from pyspark.sql import functions as F


def write_delta(df, table_name):
    """Save `df` as a managed Delta table.

    The table is replaced on every run (data and schema), so a notebook can be rerun
    from the top without leaving old rows or old columns behind. Delta keeps the
    previous versions in the table history.
    """
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))
    description = TABLE_CONTRACT.get(table_name, {}).get("description")
    if description:
        spark.sql(f"COMMENT ON TABLE {table_name} IS '{description}'")
    print("Wrote", table_name)


def _validate_table(table_name, expected_rows):
    contract = TABLE_CONTRACT.get(table_name, {})
    if not spark.catalog.tableExists(table_name):
        written_by = contract.get("written_by", "its notebook")
        return (table_name, None, expected_rows, None, None, "FAIL",
                f"table does not exist: run {written_by} first")

    saved = spark.table(table_name)                      # read back from the catalog
    storage_format = spark.sql(f"DESCRIBE DETAIL {table_name}").first()["format"]
    rows = saved.count()

    problems = []
    if storage_format != "delta":
        problems.append(f"format is {storage_format}, not delta")
    if rows == 0:
        problems.append("table is empty")
    if expected_rows is not None and rows != expected_rows:
        problems.append(f"expected {expected_rows:,} rows")
    missing = [c for c in contract.get("required_columns", []) if c not in saved.columns]
    if missing:
        problems.append("missing columns: " + ", ".join(missing))
    key = contract.get("key")
    if key and key in saved.columns:
        key_stats = saved.agg(
            F.countDistinct(key).alias("distinct_keys"),
            F.sum(F.col(key).isNull().cast("int")).alias("null_keys"),
        ).first()
        if key_stats["null_keys"]:
            problems.append(f"{key} has {key_stats['null_keys']:,} missing values")
        elif key_stats["distinct_keys"] != rows:
            problems.append(f"{key} is not unique")

    return (table_name, storage_format, expected_rows, rows, len(saved.columns),
            "FAIL" if problems else "PASS", "; ".join(problems))


def validate_tables(expectations):
    """Read tables back from the catalog and check them against the table contract.

    `expectations` maps a table name to the row count it should have, or to None
    when only the other checks apply. Example:

        validate_tables({BRONZE_TABLE: 10000, SILVER_TABLE: None})

    Checks: the table exists, is stored as Delta, is not empty, has the expected row
    count, contains the contract's required columns, and its key is unique.
    Displays one row per table and stops the notebook if any check fails.
    """
    results = [_validate_table(name, rows) for name, rows in expectations.items()]
    display(spark.createDataFrame(
        results,
        "table string, format string, expected_rows long, actual_rows long, "
        "columns int, status string, problems string"))
    failed = [r[0] for r in results if r[5] != "PASS"]
    if failed:
        raise AssertionError(f"Delta validation failed for: {failed}")

In [ ]:
# --- Shared by the model notebooks ----------------------------------------------
SPLIT_NAMES = ("train", "validation", "test")


def load_model_splits():
    """Return (train, validation, test) DataFrames built from the Gold feature table.

    Rows are assigned by joining on ID to the Phase 2 split lookup, so every model
    trains and tests on the same drivers as its scikit-learn version in Phase 2.
    Phase 2 sizes: 6,400 train, 1,600 validation, 2,000 test.
    """
    features = spark.table(GOLD_FEATURES_TABLE).drop("split")
    assignment = spark.table(SPLIT_TABLE).select(
        F.col(ID_COLUMN).cast("string").alias("_split_id"), "split")
    joined = (features
              .withColumn("_split_id", F.col(ID_COLUMN).cast("string"))
              .join(assignment, on="_split_id", how="inner")
              .drop("_split_id"))
    return tuple(joined.filter(F.col("split") == name).drop("split") for name in SPLIT_NAMES)


def evaluate_classifier(predictions, label_col=LABEL_COLUMN):
    """Score binary predictions the same way as Phase 2, for the claim class (label 1).

    `predictions` is the output of a fitted Spark ML model's transform(): it needs the
    label column plus `prediction` and `rawPrediction`. Returns a dict with accuracy,
    precision, recall, f1, roc_auc and the confusion-matrix counts tp, fp, tn, fn.
    """
    from pyspark.ml.evaluation import BinaryClassificationEvaluator

    actual, predicted = F.col(label_col) == 1, F.col("prediction") == 1
    counts = predictions.agg(
        F.sum((actual & predicted).cast("int")).alias("tp"),
        F.sum((~actual & predicted).cast("int")).alias("fp"),
        F.sum((~actual & ~predicted).cast("int")).alias("tn"),
        F.sum((actual & ~predicted).cast("int")).alias("fn"),
    ).first()
    tp, fp, tn, fn = (int(counts[name] or 0) for name in ("tp", "fp", "tn", "fn"))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    roc_auc = BinaryClassificationEvaluator(
        labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(predictions)

    return {
        "accuracy": (tp + tn) / (tp + fp + tn + fn),
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "roc_auc": roc_auc,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
    }


print(f"Setup loaded: tables in {CATALOG}.{SCHEMA}, files in {VOLUME_ROOT}")